In [ ]:
from pathlib import Path

import pandas as pd

from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR, VersionObject

DOCUMENT_REGISTRY: Path =  REGISTRIES_DIR / "document_registry.csv"
BASE_REGISTRY: Path = REGISTRIES_DIR / "base_output_registry.csv"
RAW_REGISTRY: Path = SUBREGISTRIES_DIR / "raw_file_output_registry.csv"
EXTRACTION_REGISTRY: Path = SUBREGISTRIES_DIR / "extraction_output_registry.csv"
CLASSIFICATION_REGISTRY: Path = SUBREGISTRIES_DIR / "DAS_classification_output_registry.csv"

version_object = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.4")
df_document_registry = pd.read_csv(DOCUMENT_REGISTRY)
df_base_registry = pd.read_csv(BASE_REGISTRY)
df_raw_registry = pd.read_csv(RAW_REGISTRY)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY)
df_DAS_classification_registry = pd.read_csv(CLASSIFICATION_REGISTRY)

In [ ]:
merged_df_classification = df_DAS_classification_registry.merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")

In [ ]:
duplicated = df_base_registry.drop(columns=["creation_date"]).duplicated().sum()

In [ ]:
duplicated["output_type"].value_counts()

In [ ]:
cols = df_base_registry.columns.drop("creation_date")

dup_mask = df_base_registry.duplicated(subset=cols, keep=False)

df_base_registry[dup_mask]["output_type"].value_counts()

In [ ]:
df_DAS_classification_registry.duplicated().sum()

In [ ]:
# Check the columns in merged_df_classification
print("Columns:", merged_df_classification.columns.tolist())

# Check what values exist in key columns
print("\nUnique section_type values:", merged_df_classification["section_type"].unique() if "section_type" in merged_df_classification.columns else "Column not found")
print("Unique stage values:", merged_df_classification["stage"].unique() if "stage" in merged_df_classification.columns else "Column not found")
print("Unique software_version values:", merged_df_classification["software_version"].unique() if "software_version" in merged_df_classification.columns else "Column not found")
print("\nversion_object.software_version =", version_object.software_version)

# Check the filter step-by-step
condition1 = merged_df_classification["section_type"]=="DAS"
condition2 = merged_df_classification["stage"]=="cleaned"
condition3 = merged_df_classification["software_version"]==version_object.software_version

print(f"\nRows matching condition1 (section_type=='DAS'): {condition1.sum()}")
print(f"Rows matching condition2 (stage=='cleaned'): {condition2.sum()}")
print(f"Rows matching condition3 (software_version=='{version_object.software_version}'): {condition3.sum()}")
print(f"Rows matching ALL conditions: {(condition1 & condition2 & condition3).sum()}")

In [ ]:
df_DAS_classification_registry = pd.read_csv(CLASSIFICATION_REGISTRY)
df_classifications = df_DAS_classification_registry.loc[df_DAS_classification_registry["model"]=="claude-haiku-4-5-20251001", "label"]
df_classifications.duplicated().sum()

In [ ]:
merge_base_classification = df_DAS_classification_registry.merge(df_base_registry[["output_sha", "software_version"]], on="output_sha", how="left")
df_haiku_latest = merge_base_classification.loc[merge_base_classification["software_version"]=="v1.0.2"]
df_haiku_latest.value_counts(subset="label")

In [ ]:
df_haiku_latest.duplicated().sum()

In [ ]:
merge_base_classification.duplicated().sum()

In [ ]:
# Check for duplicate output_sha in the right dataframe
print(df_base_registry["output_sha"].duplicated().sum())

# Check total rows after merge vs expected
print(f"Rows after merge: {len(merge_base_classification)}")
print(f"Rows in haiku_latest: {len(df_haiku_latest)}")

# Check for NaN labels (the missing count)
print(f"NaN labels: {df_haiku_latest['label'].isna().sum()}")

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt

counts = df_haiku_latest.value_counts(subset="label")
total = counts.sum()
pcts = (counts / total * 100).round(1)

labels = counts.index.tolist()
values = counts.values
colors = ['#888780', '#D85A30', '#1D9E75', '#7F77DD', '#FAC775', '#F09595', '#D4537E']

fig = plt.figure(figsize=(14, 5))
fig.suptitle(f"DAS classification results — v1.0.2  (n={total:,})", fontsize=13, fontweight='500', x=0.02, ha='left')
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)

# --- metric cards (text axes) ---
card_data = [
    ("total sections", f"{total:,}", "#333"),
    ("open access",    f"{pcts.get('open_access', 0):.1f}%", "#1D9E75"),
    ("no data",        f"{pcts.get('no_data', 0):.1f}%", "#888780"),
    ("restricted",     f"{pcts.get('restricted_access', 0):.1f}%", "#D85A30"),
]

ax_cards = fig.add_subplot(gs[0, 0])
ax_cards.axis('off')
for i, (label, val, color) in enumerate(card_data):
    y = 0.85 - i * 0.22
    ax_cards.text(0.05, y,      label, fontsize=9,  color='gray',  transform=ax_cards.transAxes)
    ax_cards.text(0.05, y-0.09, val,   fontsize=16, color=color, fontweight='bold', transform=ax_cards.transAxes)

# --- donut ---
ax_donut = fig.add_subplot(gs[0, 1])
wedges, _ = ax_donut.pie(values, colors=colors, startangle=90,
                          wedgeprops=dict(width=0.5, edgecolor='white', linewidth=1.5))
ax_donut.set_title("distribution", fontsize=10, color='gray', pad=10)

# --- horizontal bar ---
ax_bar = fig.add_subplot(gs[0, 2])
bars = ax_bar.barh(labels[::-1], pcts.values[::-1], color=colors[::-1], height=0.6)
ax_bar.set_xlabel("% of sections", fontsize=9, color='gray')
ax_bar.tick_params(axis='y', labelsize=9)
ax_bar.tick_params(axis='x', labelsize=9, colors='gray')
ax_bar.spines[['top', 'right', 'left']].set_visible(False)
ax_bar.xaxis.grid(True, linestyle='--', alpha=0.4)
ax_bar.set_axisbelow(True)

for bar, pct, cnt in zip(bars, pcts.values[::-1], values[::-1]):
    ax_bar.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f"{pct}%  ({cnt:,})", va='center', fontsize=8, color='gray')

ax_bar.set_xlim(0, pcts.max() + 15)

plt.tight_layout()
plt.savefig("das_classification_v102.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

has_data = df_haiku_latest[df_haiku_latest["label"] != "no_data"]
counts = has_data.value_counts(subset="label")
total = counts.sum()
pcts = (counts / total * 100).round(1)

labels = counts.index.tolist()
values = counts.values
colors = ['#D85A30', '#1D9E75', '#7F77DD', '#FAC775', '#F09595', '#D4537E']

fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle(
    f"DAS classification — studies with associated data  (n={total:,} / {len(df_haiku_latest):,} total)",
    fontsize=11, x=0.02, ha='left'
)

bars = ax.barh(labels[::-1], pcts.values[::-1], color=colors[::-1], height=0.55)

ax.set_xlabel("% of sections with data", fontsize=9, color='gray')
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9, colors='gray')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

for bar, pct, cnt in zip(bars, pcts.values[::-1], values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{pct}%  ({cnt:,})", va='center', fontsize=9, color='gray')

ax.set_xlim(0, pcts.max() + 18)

plt.tight_layout()
plt.savefig("das_classification_with_data.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

counts = df_document_registry["has_DAS"].value_counts()
total = counts.sum()
has = counts.get(True, 0)
has_not = counts.get(False, 0)

fig, ax = plt.subplots(figsize=(6, 3))
fig.suptitle(
    f"DAS presence across studies, 2025 (n={total:,})",
    fontsize=11, x=0.02, ha='left'
)

bars = ax.barh(
    ["no DAS", "has DAS"],
    [has_not / total * 100, has / total * 100],
    color=['#888780', '#1D9E75'], height=0.45
)

ax.set_xlabel("% of studies", fontsize=9, color='gray')
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9, colors='gray')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

for bar, cnt in zip(bars, [has_not, has]):
    pct = cnt / total * 100
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%  ({cnt:,})", va='center', fontsize=9, color='gray')

ax.set_xlim(0, 100 + 12)

plt.tight_layout()
plt.savefig("das_presence.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Check duplicates in each dataframe independently
print("Duplicates in df_classification_registry:", df_DAS_classification_registry["output_sha"].duplicated().sum())
print("Duplicates in df_base_registry:", df_base_registry["output_sha"].duplicated().sum())

# If df_base_registry is the culprit, see how many per sha
dup_counts = df_base_registry[df_base_registry["output_sha"].duplicated(keep=False)] \
    .groupby("output_sha") \
    .size() \
    .sort_values(ascending=False)
print(dup_counts.head(20))

# Also check if the duplicates differ in software_version 
# (i.e. same sha processed across multiple runs)
df_base_registry[df_base_registry["output_sha"].duplicated(keep=False)] \
    .groupby("output_sha")["software_version"] \
    .nunique() \
    .value_counts()

In [ ]:
dup_shas = df_base_registry[df_base_registry["output_sha"].duplicated(keep=False)]

# Case 1: same sha, different versions (reprocessing issue)
cross_version = dup_shas.groupby("output_sha").filter(
    lambda x: x["software_version"].nunique() > 1
)
print(cross_version.sort_values("output_sha").head(10))

# Case 2: same sha, same version (collision or double write)
same_version = dup_shas.groupby("output_sha").filter(
    lambda x: x["software_version"].nunique() == 1
)
print(same_version.sort_values("output_sha").head(10))

# For case 2, check if ALL columns are identical (pure double write)
# or if they differ (actual SHA collision)
same_version_dupes = same_version.drop_duplicates()
print(f"Rows remaining after full dedup: {len(same_version_dupes)}")
print(f"Original duplicate rows: {len(same_version)}")

In [ ]:
import matplotlib.pyplot as plt

df_das_context = (
    df_haiku_latest
    .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
    .merge(df_document_registry[["doc_doi", "journal", "publication_year"]], on="doc_doi", how="left")
)
df_das_context["journal"] = df_das_context["journal"].fillna("Unknown journal")

label_order = [
    "restricted_access",
    "open_access",
    "unclear",
    "partial_access",
    "no_access",
    "incorrect",
]
label_colors = {
    "restricted_access": "#D85A30",
    "open_access": "#1D9E75",
    "unclear": "#7F77DD",
    "partial_access": "#FAC775",
    "no_access": "#F09595",
    "incorrect": "#D4537E",
}

df_das_with_data = df_das_context[df_das_context["label"] != "no_data"].copy()
journal_order = df_das_with_data["journal"].value_counts().index.tolist()

das_by_journal = (
    pd.crosstab(df_das_with_data["journal"], df_das_with_data["label"], normalize="index")
    .reindex(index=journal_order)
    .reindex(columns=label_order, fill_value=0)
    * 100
).round(1)
journal_totals = df_das_with_data["journal"].value_counts().reindex(journal_order)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
left = pd.Series(0.0, index=das_by_journal.index)

for label in label_order:
    widths = das_by_journal[label]
    ax.barh(
        das_by_journal.index,
        widths,
        left=left,
        color=label_colors[label],
        edgecolor="white",
        height=0.65,
        label=label,
    )
    left = left + widths

ax.invert_yaxis()
ax.set_xlabel("% of DAS sections, excluding no_data", fontsize=9, color="gray")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", labelsize=9, colors="gray")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for journal, total in journal_totals.items():
    ax.text(101.5, journal, f"n={total:,}", va="center", fontsize=9, color="gray")

ax.set_xlim(0, 112)
ax.legend(title="DAS label", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.suptitle(
    f"DAS classification by journal, excluding no_data (n={len(df_das_with_data):,})",
    fontsize=12,
    x=0.02,
    ha="left",
)

plt.tight_layout()
plt.savefig("das_classification_by_journal_excluding_no_data.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

if "df_das_context" not in globals():
    df_das_context = (
        df_haiku_latest
        .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
        .merge(df_document_registry[["doc_doi", "journal", "publication_year"]], on="doc_doi", how="left")
    )
df_das_context["journal"] = df_das_context["journal"].fillna("Unknown journal")

if "label_order" not in globals() or "restricted_access" not in label_order:
    label_order = [
        "restricted_access",
        "open_access",
        "unclear",
        "partial_access",
        "no_access",
        "incorrect",
    ]

df_das_with_data = df_das_context[df_das_context["label"] != "no_data"].copy()
if "journal_order" not in globals():
    journal_order = df_das_with_data["journal"].value_counts().index.tolist()

das_counts_by_journal = (
    pd.crosstab(df_das_with_data["journal"], df_das_with_data["label"])
    .reindex(index=journal_order)
    .reindex(columns=label_order, fill_value=0)
)
journal_totals = das_counts_by_journal.sum(axis=1)

count_cmap = LinearSegmentedColormap.from_list(
    "das_counts",
    ["#f7f4ea", "#d7d1bc", "#9c9485", "#5c554c"],
)

fig_width = max(10, 1.4 * len(das_counts_by_journal.columns) + 3)
fig_height = max(4.8, 0.8 * len(das_counts_by_journal.index) + 1.8)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

im = ax.imshow(das_counts_by_journal.values, cmap=count_cmap, aspect="auto")

ax.set_xticks(range(len(das_counts_by_journal.columns)))
ax.set_xticklabels(das_counts_by_journal.columns, rotation=25, ha="right", fontsize=9)
ax.set_yticks(range(len(das_counts_by_journal.index)))
ax.set_yticklabels(das_counts_by_journal.index, fontsize=9)
ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

max_value = das_counts_by_journal.values.max() if das_counts_by_journal.values.size else 0
for i, journal in enumerate(das_counts_by_journal.index):
    for j, label in enumerate(das_counts_by_journal.columns):
        value = int(das_counts_by_journal.iloc[i, j])
        text_color = "white" if max_value and value > max_value * 0.45 else "#2f2a24"
        ax.text(j, i, f"{value}", ha="center", va="center", color=text_color, fontsize=9, fontweight="bold")

for i, total in enumerate(journal_totals):
    ax.text(
        len(das_counts_by_journal.columns) - 0.05,
        i,
        f" total={int(total):,}",
        ha="left",
        va="center",
        fontsize=9,
        color="gray",
        clip_on=False,
    )

ax.set_xticks([x - 0.5 for x in range(1, len(das_counts_by_journal.columns))], minor=True)
ax.set_yticks([y - 0.5 for y in range(1, len(das_counts_by_journal.index))], minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)
for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cbar.ax.tick_params(labelsize=8, colors="gray")
cbar.outline.set_visible(False)
cbar.set_label("count", fontsize=9, color="gray")

fig.suptitle("DAS classification by journal - category counts excluding no_data", fontsize=12, x=0.02, ha="left")
plt.tight_layout()
plt.savefig("das_classification_by_journal_counts_excluding_no_data.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

if "df_das_context" not in globals():
    df_das_context = (
        df_haiku_latest
        .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
        .merge(df_document_registry[["doc_doi", "journal", "publication_year"]], on="doc_doi", how="left")
    )

year_labels = df_das_context["publication_year"].astype("Int64").astype(str).replace("<NA>", "Unknown year")
year_order = sorted([label for label in year_labels.unique() if label != "Unknown year"], key=int)
if (year_labels == "Unknown year").any():
    year_order.append("Unknown year")

das_by_year = (
    pd.crosstab(year_labels, df_das_context["label"], normalize="index")
    .reindex(index=year_order)
    .reindex(columns=label_order, fill_value=0)
    * 100
).round(1)
year_totals = year_labels.value_counts().reindex(year_order)

fig, ax = plt.subplots(figsize=(10, max(4.5, 0.6 * len(year_order) + 1.5)))
left = pd.Series(0.0, index=das_by_year.index)

for label in label_order:
    widths = das_by_year[label]
    ax.barh(
        das_by_year.index,
        widths,
        left=left,
        color=label_colors[label],
        edgecolor="white",
        height=0.65,
        label=label,
    )
    left = left + widths

ax.invert_yaxis()
ax.set_xlabel("% of DAS sections", fontsize=9, color="gray")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", labelsize=9, colors="gray")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for year, total in year_totals.items():
    ax.text(101.5, year, f"n={total:,}", va="center", fontsize=9, color="gray")

ax.set_xlim(0, 110)
ax.legend(title="DAS label", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.suptitle("DAS classification by publication year", fontsize=12, x=0.02, ha="left")

plt.tight_layout()
plt.savefig("das_classification_by_year.png", dpi=150, bbox_inches="tight")
plt.show()


In [2]:
# Targeted rerun for CMS data-preservation policy DAS statements
#
# Purpose:
# - Select only the DAS sections whose text is the CMS data preservation/reuse/open-access policy statement.
# - Reclassify them after updating classification_schema_DAS.yaml.
# - Write new classification rows under a new software version while preserving the existing cleaned-section dependency.
#
# Safety:
# - DRY_RUN=True means this cell only previews selected rows and writes nothing.
# - Set DRY_RUN=False only after editing the schema and confirming the preview.

from __future__ import annotations

import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

import anthropic
import pandas as pd
import regex
from tqdm import tqdm

from application.utils import compute_hashes, load_classification_schema_DAS, save_registry
from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR, VersionObject, resolve_registry_path

DRY_RUN = False
FORCE_PROCESSING = False
TARGET_VERSION = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.5")
CLAUDE_MODEL = "claude-haiku-4-5-20251001"
CHECKPOINT_INTERVAL = 25

BASE_REGISTRY = REGISTRIES_DIR / "base_output_registry.csv"
EXTRACTION_REGISTRY = SUBREGISTRIES_DIR / "extraction_output_registry.csv"
DAS_CLASSIFICATION_REGISTRY = SUBREGISTRIES_DIR / "DAS_classification_output_registry.csv"

CMS_POLICY_TEXTS = {
    "Release and preservation of data used by the CMS Collaboration as the basis for publications is guided by the CMS data preservation, reuse and open access policy.",
    "Release and preservation of data used by the CMS Collaboration as the basis for publications is guided by the CMS data preservation, reuse, and open access policy.",
}

# Keep this False for the narrow rerun. Flip to True only if you also want citation/prefix variants such as
# "—Release ... open access policy [121]." to be included.
INCLUDE_CMS_POLICY_REGEX_VARIANTS = True
CMS_POLICY_REGEX = re.compile(
    r"^[-—–.\s]*Release and preservation of data used by the CMS [Cc]ollaboration as the basis for publications "
    r"is guided by the CMS (?:policy as stated in the )?data preservation, reuse,? and open access policy(?:\s*\[\d+\])?\.?$"
)


def normalize_statement_text(value: object) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def is_target_cms_policy_statement(value: object) -> bool:
    normalized = normalize_statement_text(value)
    if normalized in CMS_POLICY_TEXTS:
        return True
    return INCLUDE_CMS_POLICY_REGEX_VARIANTS and bool(CMS_POLICY_REGEX.match(normalized))


def build_das_prompt(classification_schema: object, label_list: list[str], confidence_levels: dict[str, str]) -> str:
    return f"""Your task it to classify data availability statements in scientific papers according to the classification schema provided below. Some general information about the classification task:
                The following are the existing labels for the task: {label_list}.
                You must only classify sections using these labels. Please provide a level of confidence in the classification alongside the classification label.
                The following are the levels of confidence you should use to score confidence in the classification alongisde their definitions: {confidence_levels}.
                Use ONLY the keys as labels for the final classification output.
                Remember that if a section mentions information about data availability, that takes precedence over information about code availability.
                Therefore, if a section mentions information about data, that information should guide classification whereas information about code should be ignored.
                The following are priority rules you should use EXCLUSIVELY when a section mentions information about availability of multiple elements of research data under DIFFERENT access conditions
                (e.g. a section mentioning that some elements are openly available, while some are available upon request):
                - If at least one element of research data is mentioned as openly available, the section should be classified as "partial_access".
                - If at least one element of research data is mentioned as being available upon request AND no other element is mentioned as being openly available, the section should be classified as "restricted_access".
                - If at least one element of the research study materials mention information about data availability, that should take precedence over information about code availability, which should instead be ignored.
                - Information about availability of code/software should only be considered when NOT provided alongside information about data availability. In that case, the section should always be classified as "incorrect".
                Remember also that information about code/software used to generate data should be considered information about code, not information about data.
                Remembers that a large variety of research outputs can be considered as "elements of research data". That should include every research output that can be interpreted and understood by sufficiently knowledgeable humans.
                Examples of elements of research data are: numerical values, measurements, graphs, tables, images, equations, etc. Code should never be considered as an element of research data for this task.
                Inside the classification schema, you will find examples of hard positives and hard negatives for each category. Hard positives are pair of statements belonging to the SAME category that are difficult
                to classify as part of the same category. Hard negatives are pairs of statements belonging to DIFFERENT categories that are difficult to classify as being part of different categories. In the hard negative cases,
                the FIRST statement is the one belonging to the relevant category, while the SECOND statement is the one belonging to a different category.
                The following is the classification schema you should use for classifying the section. Read the classification schema carefully. Reason about how you response follows the rules provided in the schema.
                "{classification_schema}".
                Respond ONLY with a JSON object with two keys "label" and "confidence"."""


def build_target_rows(
    df_base: pd.DataFrame,
    df_extraction: pd.DataFrame,
    df_das_classification: pd.DataFrame,
) -> pd.DataFrame:
    base_lookup = df_base.drop_duplicates("output_sha", keep="last")
    extraction_lookup = df_extraction.drop_duplicates("output_sha", keep="last").rename(
        columns={"output_sha": "cleaned_dependency_sha", "text": "cleaned_text"}
    )

    classification_context = df_das_classification.merge(
        base_lookup[["output_sha", "doc_doi", "pipeline_version", "software_version", "dependencies"]],
        on="output_sha",
        how="left",
    ).rename(columns={"software_version": "source_software_version", "dependencies": "cleaned_dependency_sha"})

    targets = classification_context[
        classification_context["label"].eq("unclear")
        & classification_context["text"].map(is_target_cms_policy_statement)
    ].copy()

    targets = targets.merge(
        extraction_lookup[["cleaned_dependency_sha", "section_type", "stage", "file_path", "cleaned_text"]],
        on="cleaned_dependency_sha",
        how="left",
    )
    targets = targets[targets["section_type"].eq("DAS") & targets["stage"].eq("cleaned")].copy()

    # One rerun per document. If the same document appears in multiple source software versions, keep the latest source row.
    targets = targets.sort_values(["doc_doi", "source_software_version"]).drop_duplicates("doc_doi", keep="last")
    return targets.sort_values("doc_doi").reset_index(drop=True)


def classify_das_with_existing_dependency(
    *,
    cleaned_das_path: Path,
    doc_doi: str,
    cleaned_dependency_sha: str,
    df_base: pd.DataFrame,
    df_das_classification: pd.DataFrame,
    version_object: VersionObject,
    dependency_set: set[str],
    force_processing: bool = False,
    claude_model: str = CLAUDE_MODEL,
    api_key: str | None = None,
    retries: int = 3,
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    if api_key is None:
        api_key = os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY not found")

    if cleaned_dependency_sha in dependency_set and not force_processing:
        print(f"Skipping {doc_doi}; already classified for {version_object.software_version}")
        return df_base, df_das_classification, cleaned_dependency_sha

    if force_processing:
        old_sha = df_das_classification.loc[
            df_das_classification["method"].eq("LLM")
            & df_das_classification["output_sha"].isin(
                df_base.loc[
                    df_base["doc_doi"].eq(doc_doi)
                    & df_base["pipeline_version"].eq(version_object.pipeline_version)
                    & df_base["software_version"].eq(version_object.software_version)
                    & df_base["output_type"].eq("classification"),
                    "output_sha",
                ]
            ),
            "output_sha",
        ]
        df_base = df_base[~df_base["output_sha"].isin(old_sha)]
        df_das_classification = df_das_classification[~df_das_classification["output_sha"].isin(old_sha)]

    label_list = ["no_data", "open_access", "partial_access", "restricted_access", "no_access", "incorrect", "unclear"]
    confidence_levels = {
        "not confident at all": "the section is highly ambiguous and could arguably belong to different categories",
        "somewhat not confident": "the section has ambiguous elements, but one category seems slightly more correct than others",
        "neither confident nor not confident": "the section has some ambiguous elements but it could reasonably be classified as belonging to a category",
        "somewhat confident": "the section reasonably fits one category, but has some minor ambiguities",
        "very confident": "the section clearly satisfies a strong logical condition for classifying it into one category",
    }
    prompt = build_das_prompt(load_classification_schema_DAS(), label_list, confidence_levels)
    json_pattern = regex.compile(r"\{.*?\}", regex.DOTALL)
    das_text = cleaned_das_path.read_text()
    client = anthropic.Anthropic(api_key=api_key)

    print(f"Starting targeted DAS classification for {doc_doi}")
    for attempt in range(retries):
        try:
            response = client.messages.create(
                model=claude_model,
                max_tokens=500,
                temperature=0,
                system=[{"type": "text", "text": prompt, "cache_control": {"type": "ephemeral"}}],
                messages=[{"role": "user", "content": f"classify this data availability statement:{das_text}"}],
            )
            time.sleep(0.5)
            response_text = response.content[0].text.strip()
            json_match = json_pattern.search(response_text)
            if not json_match:
                print(f"No JSON found for {doc_doi}: {response_text}")
                return df_base, df_das_classification, cleaned_dependency_sha

            classification_result = json.loads(json_match.group())
            category_label = classification_result["label"]
            confidence_level = classification_result["confidence"]
            if category_label not in label_list or confidence_level not in confidence_levels:
                print(f"Invalid classification for {doc_doi}: label={category_label}, confidence={confidence_level}")
                return df_base, df_das_classification, cleaned_dependency_sha

            output_sha = compute_hashes(
                cleaned_das_path,
                salt=f"classified:{doc_doi}_{claude_model}_{version_object.software_version}",
            )
            base_row = {
                "output_sha": output_sha,
                "doc_doi": doc_doi,
                "output_type": "classification",
                "pipeline_version": version_object.pipeline_version,
                "software_version": version_object.software_version,
                "creation_date": datetime.now(),
                "dependencies": cleaned_dependency_sha,
            }
            classification_row = {
                "output_sha": output_sha,
                "label": category_label,
                "method": "LLM",
                "model": claude_model,
                "confidence": confidence_level,
                "text": das_text,
            }
            print(f"Writing targeted row for {doc_doi}, label={category_label}")
            df_base = pd.concat([df_base, pd.DataFrame([base_row])], ignore_index=True)
            df_das_classification = pd.concat([df_das_classification, pd.DataFrame([classification_row])], ignore_index=True)
            return df_base, df_das_classification, cleaned_dependency_sha
        except anthropic.RateLimitError:
            wait = 2**attempt
            print(f"Rate limit hit for {doc_doi}; retrying in {wait}s")
            time.sleep(wait)
        except anthropic.APIError as exc:
            print(f"API error for {doc_doi}: {exc}")
            return df_base, df_das_classification, cleaned_dependency_sha
        except Exception as exc:
            print(f"Unexpected error for {doc_doi}: {exc}")
            return df_base, df_das_classification, cleaned_dependency_sha

    print(f"Max retries exceeded for {doc_doi}")
    return df_base, df_das_classification, cleaned_dependency_sha


df_base_registry = pd.read_csv(BASE_REGISTRY)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY)
df_DAS_classification_registry = pd.read_csv(DAS_CLASSIFICATION_REGISTRY)

target_rows = build_target_rows(df_base_registry, df_extraction_registry, df_DAS_classification_registry)
print(f"Target documents selected: {len(target_rows):,}")
display(target_rows[["doc_doi", "source_software_version", "cleaned_dependency_sha", "file_path", "text"]].head(20))
print(target_rows["text"].map(normalize_statement_text).value_counts().to_string())

if DRY_RUN:
    print("DRY_RUN=True, so no API calls or registry writes were made. Set DRY_RUN=False after updating the schema.")
else:
    existing_dependency_set = set(
        df_base_registry.loc[
            df_base_registry["output_type"].eq("classification")
            & df_base_registry["pipeline_version"].eq(TARGET_VERSION.pipeline_version)
            & df_base_registry["software_version"].eq(TARGET_VERSION.software_version),
            "dependencies",
        ].dropna()
    )

    rows_since_checkpoint = 0
    for _, row in tqdm(target_rows.iterrows(), total=len(target_rows), desc="Targeted CMS DAS rerun"):
        df_base_registry, df_DAS_classification_registry, dependency_sha = classify_das_with_existing_dependency(
            cleaned_das_path=resolve_registry_path(row["file_path"]),
            doc_doi=row["doc_doi"],
            cleaned_dependency_sha=row["cleaned_dependency_sha"],
            df_base=df_base_registry,
            df_das_classification=df_DAS_classification_registry,
            version_object=TARGET_VERSION,
            dependency_set=existing_dependency_set,
            force_processing=FORCE_PROCESSING,
            claude_model=CLAUDE_MODEL,
        )
        existing_dependency_set.add(dependency_sha)
        rows_since_checkpoint += 1
        if CHECKPOINT_INTERVAL > 0 and rows_since_checkpoint >= CHECKPOINT_INTERVAL:
            save_registry(df_base_registry, BASE_REGISTRY)
            save_registry(df_DAS_classification_registry, DAS_CLASSIFICATION_REGISTRY)
            rows_since_checkpoint = 0

    save_registry(df_base_registry, BASE_REGISTRY)
    save_registry(df_DAS_classification_registry, DAS_CLASSIFICATION_REGISTRY)
    print(f"Finished targeted CMS DAS rerun under software_version={TARGET_VERSION.software_version}")


Target documents selected: 58


,doc_doi,source_software_version,cleaned_dependency_sha,file_path,text
0,10.1007/JHEP02(2025)036,v1.0.2,ed5ea791fc10e1ce8f4c7b8cfa022954ab9c4d24,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...
1,10.1007/JHEP02(2025)040,v1.0.2,e3e8f7a6bc50b434c3625093627570cb125b0761,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...
2,10.1007/JHEP02(2025)050,v1.0.2,ebafaa05e28429e68e62b2bddea763ecd411ad2b,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...
3,10.1007/JHEP02(2025)064,v1.0.2,1177b1a2113d79f5ef1444e437391da607f2968a,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...
4,10.1007/JHEP02(2025)097,v1.0.2,c6fbda8da86461dd0d14a93eedcd1405cb3a451f,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...
5,10.1007/JHEP02(2025)177,v1.0.2,d1159174aa63eb685fcf50717c388b1fac38563e,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...
6,10.1007/JHEP02(2025)195,v1.0.2,c6dd89c859b6158eeea0d1e502f9ce5ed1b8f38f,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...
7,10.1007/JHEP02(2025)199,v1.0.2,d51fbd9646abdee835a02d798eb582dc5dd7b400,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...
8,10.1007/JHEP03(2025)114,v1.0.2,a7dfd7cb0dfa40502230d4e2d060cad9ce6a9fd1,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...
9,10.1007/JHEP04(2025)099,v1.0.2,27ed30518c883a782aef0e47d5286d5e1600232f,data/records/software_version/v1.0.2/pipeline_...,Release and preservation of data used by the C...


text
Release and preservation of data used by the CMS Collaboration as the basis for publications is guided by the CMS data preservation, reuse and open access policy.            33
Release and preservation of data used by the CMS Collaboration as the basis for publications is guided by the CMS data preservation, reuse, and open access policy.           14
Release and preservation of data used by the CMS collaboration as the basis for publications is guided by the CMS data preservation, reuse, and open access policy.            1
Release and preservation of data used by the CMS collaboration as the basis for publications is guided by the CMS data preservation, reuse and open access policy.             1
—Release and preservation of data used by the CMS Collaboration as the basis for publications is guided by the CMS data preservation, reuse, and open access policy [121].     1
—Release and preservation of data used by the CMS Collaboration as the basis for publications is guided by the

Targeted CMS DAS rerun:   0%|          | 0/58 [00:00<?, ?it/s]

Skipping 10.1007/JHEP02(2025)036; already classified for v1.0.5
Skipping 10.1007/JHEP02(2025)040; already classified for v1.0.5
Skipping 10.1007/JHEP02(2025)050; already classified for v1.0.5
Skipping 10.1007/JHEP02(2025)064; already classified for v1.0.5
Skipping 10.1007/JHEP02(2025)097; already classified for v1.0.5
Skipping 10.1007/JHEP02(2025)177; already classified for v1.0.5
Skipping 10.1007/JHEP02(2025)195; already classified for v1.0.5
Skipping 10.1007/JHEP02(2025)199; already classified for v1.0.5
Skipping 10.1007/JHEP03(2025)114; already classified for v1.0.5
Starting targeted DAS classification for 10.1007/JHEP04(2025)099


...

Targeted CMS DAS rerun:  93%|█████████▎| 54/58 [00:33<00:06,  1.75s/it]

Writing targeted row for 10.1103/srvm-f1h3, label=open_access
Starting targeted DAS classification for 10.1103/zc76-rgcp


Targeted CMS DAS rerun: 100%|██████████| 58/58 [00:36<00:00,  1.58it/s]

Writing targeted row for 10.1103/zc76-rgcp, label=open_access
Skipping 10.1140/epjc/s10052-024-13606-8; already classified for v1.0.5
Skipping 10.1140/epjc/s10052-024-13729-y; already classified for v1.0.5
Skipping 10.1140/epjc/s10052-025-14713-w; already classified for v1.0.5


Finished targeted CMS DAS rerun under software_version=v1.0.5
